In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\| **Autograd** \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

学习基础知识 \|\| 快速入门 \|\| 张量 \|\| 数据集与数据加载器 \|\| 数据变换 \|\| 构建模型 \|\| **自动微分** \|\| 优化模型参数 \|\| 保存与加载模型

Automatic Differentiation with `torch.autograd` 使用 `torch.autograd` 自动微分
===============================================

When training neural networks, the most frequently used algorithm is
**back propagation**. In this algorithm, parameters (model weights) are
adjusted according to the **gradient** of the loss function with respect
to the given parameter.

训练神经网络时，最常用的算法是**反向传播**。在该算法中，参数（模型权重）会根据损失函数关于该参数的**梯度**进行调整。

To compute those gradients, PyTorch has a built-in differentiation
engine called `torch.autograd`. It supports automatic computation of
gradient for any computational graph.

为了计算这些梯度，PyTorch 内置了一个名为 `torch.autograd` 的微分引擎。它支持对任意计算图自动计算梯度。

Consider the simplest one-layer neural network, with input `x`,
parameters `w` and `b`, and some loss function. It can be defined in
PyTorch in the following manner:

考虑最简单的单层神经网络：输入为 `x`，参数为 `w` 和 `b`，再加上某个损失函数。在 PyTorch 中可以这样定义它：


In [ ]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

Tensors, Functions and Computational graph 张量、函数与计算图
==========================================

This code defines the following **computational graph**:

这段代码定义了下面的**计算图**：

![](https://pytorch.org/tutorials/_static/img/basics/comp-graph.png)

In this network, `w` and `b` are **parameters**, which we need to
optimize. Thus, we need to be able to compute the gradients of loss
function with respect to those variables. In order to do that, we set
the `requires_grad` property of those tensors.

在这个网络中，`w` 和 `b` 是需要优化的**参数**。因此，我们需要能够计算损失函数关于这些变量的梯度。为此，我们设置这些张量的 `requires_grad` 属性。


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE 注意：</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>You can set the value of <code>requires_grad</code> when creating atensor, or later by using <code>x.requires_grad_(True)</code> method.</p>

<p>你可以在创建张量时设置 <code>requires_grad</code> 的值，也可以之后使用 <code>x.requires_grad_(True)</code> 方法设置。</p>

</div>



A function that we apply to tensors to construct computational graph is
in fact an object of class `Function`. This object knows how to compute
the function in the *forward* direction, and also how to compute its
derivative during the *backward propagation* step. A reference to the
backward propagation function is stored in `grad_fn` property of a
tensor. You can find more information of `Function` [in the
documentation](https://pytorch.org/docs/stable/autograd.html#function).

我们作用在张量上以构建计算图的函数，实际上是 `Function` 类的一个对象。这个对象知道如何在*前向*方向上计算函数，也知道如何在*反向传播*步骤中计算它的导数。反向传播函数的引用存储在张量的 `grad_fn` 属性中。你可以[在文档中](https://pytorch.org/docs/stable/autograd.html#function)找到关于 `Function` 的更多信息。


In [ ]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Computing Gradients 计算梯度
===================

To optimize weights of parameters in the neural network, we need to
compute the derivatives of our loss function with respect to parameters,
namely, we need $\frac{\partial loss}{\partial w}$ and
$\frac{\partial loss}{\partial b}$ under some fixed values of `x` and
`y`. To compute those derivatives, we call `loss.backward()`, and then
retrieve the values from `w.grad` and `b.grad`:

为了优化神经网络中参数的权重，我们需要计算损失函数关于参数的导数，也就是说，在 `x` 和 `y` 取某些固定值时，我们需要 $\frac{\partial loss}{\partial w}$ 和 $\frac{\partial loss}{\partial b}$。为了计算这些导数，我们调用 `loss.backward()`，然后从 `w.grad` 和 `b.grad` 中取出这些值：


In [ ]:
loss.backward()
print(w.grad)
print(b.grad)

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE 注意：</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<ul>
<li>We can only obtain the <code>grad</code> properties for the leafnodes of the computational graph, which have <code>requires_grad</code> propertyset to <code>True</code>. For all other nodes in our graph, gradients will not beavailable.- We can only perform gradient calculations using<code>backward</code> once on a given graph, for performance reasons. If we needto do several <code>backward</code> calls on the same graph, we need to pass<code>retain_graph=True</code> to the <code>backward</code> call.</li>
</ul>
```

<ul>
<li>我们只能获取计算图中 <code>requires_grad</code> 属性设为 <code>True</code> 的叶节点的 <code>grad</code> 属性。图中其他节点的梯度不可用。</li>
<li>出于性能考虑，在一个给定的图上只能用 <code>backward</code> 进行一次梯度计算。如果需要在同一个图上多次调用 <code>backward</code>，需要在调用 <code>backward</code> 时传入 <code>retain_graph=True</code>。</li>
</ul>

</div>



Disabling Gradient Tracking 禁用梯度跟踪
===========================

By default, all tensors with `requires_grad=True` are tracking their
computational history and support gradient computation. However, there
are some cases when we do not need to do that, for example, when we have
trained the model and just want to apply it to some input data, i.e. we
only want to do *forward* computations through the network. We can stop
tracking computations by surrounding our computation code with
`torch.no_grad()` block:

默认情况下，所有 `requires_grad=True` 的张量都会跟踪它们的计算历史，并支持梯度计算。但在某些情况下我们不需要这样做，例如模型训练好后，只想把它应用到一些输入数据上，也就是只想通过网络做*前向*计算。我们可以把计算代码放在 `torch.no_grad()` 块中，以停止跟踪计算：


In [ ]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

Another way to achieve the same result is to use the `detach()` method
on the tensor:

实现同样效果的另一种方式是对张量使用 `detach()` 方法：


In [ ]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

There are reasons you might want to disable gradient tracking:

你可能出于以下原因想要禁用梯度跟踪：

:   -   To mark some parameters in your neural network as **frozen
        parameters**.
    -   To **speed up computations** when you are only doing forward
        pass, because computations on tensors that do not track
        gradients would be more efficient.

*   把神经网络中的某些参数标记为**冻结参数**。
*   在只做前向传播时**加速计算**，因为对不跟踪梯度的张量进行计算会更高效。


More on Computational Graphs 深入了解计算图
============================

Conceptually, autograd keeps a record of data (tensors) and all executed
operations (along with the resulting new tensors) in a directed acyclic
graph (DAG) consisting of
[Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function)
objects. In this DAG, leaves are the input tensors, roots are the output
tensors. By tracing this graph from roots to leaves, you can
automatically compute the gradients using the chain rule.

从概念上讲，autograd 在一个由 [Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function) 对象组成的有向无环图（DAG）中，记录数据（张量）以及所有执行过的操作（以及由此产生的新张量）。在这个 DAG 中，叶节点是输入张量，根节点是输出张量。从根到叶追踪这个图，就可以利用链式法则自动计算梯度。

In a forward pass, autograd does two things simultaneously:

在前向传播中，autograd 会同时做两件事：

-   run the requested operation to compute a resulting tensor
-   maintain the operation's *gradient function* in the DAG.

*   执行所请求的操作，计算出结果张量；
*   在 DAG 中维护该操作的*梯度函数*。

The backward pass kicks off when `.backward()` is called on the DAG
root. `autograd` then:

当在 DAG 的根上调用 `.backward()` 时，反向传播开始。随后 `autograd` 会：

-   computes the gradients from each `.grad_fn`,
-   accumulates them in the respective tensor's `.grad` attribute
-   using the chain rule, propagates all the way to the leaf tensors.

*   根据每个 `.grad_fn` 计算梯度；
*   把梯度累加到对应张量的 `.grad` 属性中；
*   利用链式法则，一直传播到叶张量。

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE 注意：</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>An important thing to note is that the graph is recreated from scratch; after each<code>.backward()</code> call, autograd starts populating a new graph. This isexactly what allows you to use control flow statements in your model;you can change the shape, size and operations at every iteration ifneeded.</p>

<p>需要特别注意的是，计算图是从头重新创建的：每次调用 <code>.backward()</code> 之后，autograd 都会开始构建一个新的图。正因如此，你才可以在模型中使用控制流语句；如果需要，你可以在每次迭代中改变形状、大小和操作。</p>

</div>



Optional Reading: Tensor Gradients and Jacobian Products 选读：张量梯度与雅可比积
========================================================

In many cases, we have a scalar loss function, and we need to compute
the gradient with respect to some parameters. However, there are cases
when the output function is an arbitrary tensor. In this case, PyTorch
allows you to compute so-called **Jacobian product**, and not the actual
gradient.

在很多情况下，我们有一个标量损失函数，需要计算它关于某些参数的梯度。但也有一些情况下，输出函数是任意的张量。这时，PyTorch 允许你计算所谓的**雅可比积（Jacobian product）**，而不是实际的梯度。

For a vector function $\vec{y}=f(\vec{x})$, where
$\vec{x}=\langle x_1,\dots,x_n\rangle$ and
$\vec{y}=\langle y_1,\dots,y_m\rangle$, a gradient of $\vec{y}$ with
respect to $\vec{x}$ is given by **Jacobian matrix**:

对于向量函数 $\vec{y}=f(\vec{x})$，其中 $\vec{x}=\langle x_1,\dots,x_n\rangle$，$\vec{y}=\langle y_1,\dots,y_m\rangle$，$\vec{y}$ 关于 $\vec{x}$ 的梯度由**雅可比矩阵（Jacobian matrix）**给出：

$$\begin{aligned}
J=\left(\begin{array}{ccc}
   \frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{1}}{\partial x_{n}}\\
   \vdots & \ddots & \vdots\\
   \frac{\partial y_{m}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
   \end{array}\right)
\end{aligned}$$

Instead of computing the Jacobian matrix itself, PyTorch allows you to
compute **Jacobian Product** $v^T\cdot J$ for a given input vector
$v=(v_1 \dots v_m)$. This is achieved by calling `backward` with $v$ as
an argument. The size of $v$ should be the same as the size of the
original tensor, with respect to which we want to compute the product:

PyTorch 不计算雅可比矩阵本身，而是允许你对给定的输入向量 $v=(v_1 \dots v_m)$ 计算**雅可比积** $v^T\cdot J$。这可以通过以 $v$ 为参数调用 `backward` 来实现。$v$ 的大小应与我们要对其计算乘积的原始张量的大小相同：


In [ ]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

Notice that when we call `backward` for the second time with the same
argument, the value of the gradient is different. This happens because
when doing `backward` propagation, PyTorch **accumulates the
gradients**, i.e. the value of computed gradients is added to the `grad`
property of all leaf nodes of computational graph. If you want to
compute the proper gradients, you need to zero out the `grad` property
before. In real-life training an *optimizer* helps us to do this.

注意，当我们用相同的参数第二次调用 `backward` 时，梯度的值不同了。这是因为在进行 `backward` 传播时，PyTorch 会**累加梯度**，即计算出的梯度值会加到计算图所有叶节点的 `grad` 属性上。如果想计算正确的梯度，需要先把 `grad` 属性清零。在实际训练中，*优化器*会帮我们做这件事。


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE 注意：</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Previously we were calling <code>backward()</code> function withoutparameters. This is essentially equivalent to calling<code>backward(torch.tensor(1.0))</code>, which is a useful way to compute thegradients in case of a scalar-valued function, such as loss duringneural network training.</p>

<p>之前我们调用 <code>backward()</code> 函数时没有传参数。这本质上等价于调用 <code>backward(torch.tensor(1.0))</code>，这是在标量函数（例如神经网络训练中的损失）情况下计算梯度的一种有用方式。</p>

</div>



------------------------------------------------------------------------


Further Reading 延伸阅读
===============

-   [Autograd
    Mechanics](https://pytorch.org/docs/stable/notes/autograd.html)

*   [Autograd 机制](https://pytorch.org/docs/stable/notes/autograd.html)
